# Deep width sweep on Colab

This runs the same float64 Langevin sampler as the cluster, for width 256 (tasks 9–11) and width 4,096 (tasks 21–23), three datasets, four chains, and $h/\sigma^2=0.16$. Each finished task is committed and pushed before the next one starts.

The output goes to `results/deep_width_colab`. The queued job `12681973` writes `results/deep_width` on the A10. Leave that job queued. Its seconds-per-step are the A10 measurement. This notebook records whichever GPU Colab assigns, and those seconds are for that GPU.

A width-4,096 task runs a 128-step check and then continues in the same runtime. If that check fails, only that task stops. The other tasks assigned to this runtime still run, and other notebooks are separate.

## Before you run

1. Choose **Runtime → Change runtime type → GPU**. An A100 is the better choice for width 4,096. A T4 can run one width-4,096 task, and it can take several hours.
2. Create a GitHub personal access token with access to this repository. In Colab, open the key icon in the left sidebar and add a secret named `GITHUB_TOKEN`.
3. One notebook has one GPU, so its tasks run one after another. For parallel runs, open this notebook again in another tab (**File → Open notebook in new tab**, or paste the Colab link again) and give each tab a different `TASKS` value and its own GPU runtime:
   - Tab A: `9,10,11` (width 256, all three datasets)
   - Tab B: `21` (width 4,096, dataset 0)
   - Tab C: `22` (width 4,096, dataset 1)
   - Tab D: `23` (width 4,096, dataset 2)

   A free Colab account often allows only one GPU at a time. In that case use one tab and leave `TASKS` as `9,10,11,21,22,23`. Width 256 finishes first and is already on GitHub if the later task is interrupted.

4. Run the cells in order. After every tab has pushed, run the report cell in one tab.

## Step 1. Install the checkout

This clones the repository, leaves the preinstalled Colab PyTorch in place, and installs the small Python packages the sampler imports.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

root = Path("/content/Width-Scaling-Posterior-Geometry")
remote = "https://github.com/NalinTiwary/Width-Scaling-Posterior-Geometry.git"
if root.exists():
    subprocess.check_call(["git", "pull", "--ff-only"], cwd=root)
else:
    subprocess.check_call(["git", "clone", remote, str(root)])
os.chdir(root)
sys.path[:0] = [str(root), str(root / "src"), str(root / "colab")]
os.environ["PYTHONPATH"] = f"{root}:{root / 'src'}:{root / 'colab'}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py", "pyyaml", "pandas", "scipy", "matplotlib"])
subprocess.check_call(["git", "config", "user.email", "nalint2@illinois.edu"], cwd=root)
subprocess.check_call(["git", "config", "user.name", "NalinTiwaryUIUC"], cwd=root)
print("Checkout", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## Step 2. Choose the tasks for this runtime and run them

Edit `TASKS` in the next cell before running it. Each successful task is committed and pushed immediately. Checkpoints and raw HDF5 stay on this runtime and are not pushed. The metrics CSV, run JSON, clock, and input manifest are what gets analyzed from git.

A failed task prints `TASK_FAILED` and this runtime continues with the next id.

In [ ]:
import os
import subprocess
import sys

TASKS = "9,10,11,21,22,23"

try:
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
except Exception as exc:
    raise SystemExit("Add a Colab secret named GITHUB_TOKEN, then run this cell again.") from exc

subprocess.check_call([sys.executable, "colab/run_deep_width.py", "--tasks", TASKS,
                       "--out", "results/deep_width_colab"])


## Step 3. Build the figure after the tasks you want are on GitHub

Run this once, in one tab, after the other tabs have finished. It pulls their files, writes the throughput figure, and pushes the report. Width 4,096 stays an open marker until a same-target slice reference exists. The ESS and the seconds per step are still in `probe_metrics.csv`.

In [ ]:
import os
import subprocess
import sys

if not os.environ.get("GITHUB_TOKEN"):
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
subprocess.check_call([sys.executable, "colab/run_deep_width.py", "--skip-run", "--report",
                       "--out", "results/deep_width_colab"])
